# 1. 读取单细胞参考与空间数据

先读取单细胞与空间组学之前的处理结果，并将其综合起来准备进入反卷积

In [1]:
from pathlib import Path
from IPython.display import display
import scanpy as sc
import squidpy as sq

project_dir = Path("../../")

sc_path = project_dir / "data/processed/CID4535_baseline_v1/single_cell_analysis.h5ad"
spatial_path = project_dir / "data/processed/CID4535_spatial/spatial_qc_filtering.h5ad"


adata_sc = sc.read_h5ad(sc_path)
adata_sp = sc.read_h5ad(spatial_path)


In [2]:
# 确定一下adata中的列

display(adata_sc.obs.columns.tolist())
display(adata_sp.obs.columns.to_list())

['orig.ident',
 'nCount_RNA',
 'nFeature_RNA',
 'percent.mito',
 'subtype',
 'celltype_subset',
 'celltype_minor',
 'celltype_major',
 'n_genes_by_counts',
 'total_counts',
 'total_counts_mt',
 'pct_counts_mt',
 'leiden']

['in_tissue',
 'array_row',
 'array_col',
 'pxl_row_in_fullres',
 'pxl_col_in_fullres',
 'nCount_RNA',
 'nFeature_RNA',
 'subtype',
 'patientid',
 'Classification',
 'in_tissue_label',
 'n_genes_by_counts',
 'log1p_n_genes_by_counts',
 'total_counts',
 'log1p_total_counts',
 'pct_counts_in_top_50_genes',
 'pct_counts_in_top_100_genes',
 'pct_counts_in_top_200_genes',
 'pct_counts_in_top_500_genes',
 'total_counts_mt',
 'log1p_total_counts_mt',
 'pct_counts_mt',
 'review_outside_tissue',
 'review_artefact',
 'review_missing_pathology',
 'pathology_display',
 'review_outside_tissue_display',
 'review_artefact_display',
 'review_missing_pathology_display',
 'exclude_outside_tissue',
 'exclude_artefact',
 'exclude_from_analysis',
 'exclude_from_pathology_analysis']

# 2. 查看作者标签层级与细胞数量

主要查看原先作者提供的数据的标签中celltype_major与celltype_minor的对应关系，检查各细分类包含多少细胞，再查看 CAF 和 T/NK 相关标签。  
因为原作者已经提供了细胞的注释信息，因此不使用之前leiden的聚类分析结果，直接使用作者提供的注释信息

In [3]:
# 统计所有label下的细胞数量
label_counts = (
    adata_sc.obs.groupby(["celltype_major", "celltype_minor"], dropna=False, observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
    .sort_values(
        ["celltype_major", "n_cells"],
        ascending=[True, False],
    )
)
display(label_counts)

target_cells = adata_sc.obs.loc[adata_sc.obs["celltype_major"].isin(["CAFs", "PVL", "T-cells"])]

annotation_columns = [
    "celltype_major",
    "celltype_minor",
    "celltype_subset",
]

subset_counts = (
    target_cells.groupby(annotation_columns, observed=True, dropna=False)
    .size()
    .rename("n_cells")
    .reset_index()
    .sort_values(
        ["celltype_major", "celltype_minor", "n_cells"],
        ascending=[True, True, False],
    )
)

display(subset_counts)

,celltype_major,celltype_minor,n_cells
0,B-cells,B cells Memory,56
1,CAFs,CAFs MSC iCAF-like,58
2,CAFs,CAFs myCAF-like,44
6,Cancer Epithelial,Cancer LumB SC,2025
4,Cancer Epithelial,Cancer Cycling,195
5,Cancer Epithelial,Cancer Her2 SC,2
3,Cancer Epithelial,Cancer Basal SC,1
8,Endothelial,Endothelial CXCL12,128
7,Endothelial,Endothelial ACKR1,44
10,Endothelial,Endothelial RGS5,42


,celltype_major,celltype_minor,celltype_subset,n_cells
1,CAFs,CAFs MSC iCAF-like,CAFs MSC iCAF-like s2,46
0,CAFs,CAFs MSC iCAF-like,CAFs MSC iCAF-like s1,12
4,CAFs,CAFs myCAF-like,CAFs myCAF like s5,22
3,CAFs,CAFs myCAF-like,CAFs myCAF like s4,18
2,CAFs,CAFs myCAF-like,CAFs Transitioning s3,4
5,PVL,Cycling PVL,Cycling PVL,10
6,PVL,PVL Differentiated,PVL Differentiated s3,430
8,PVL,PVL Immature,PVL_Immature s2,79
7,PVL,PVL Immature,PVL Immature s1,73
9,T-cells,Cycling T-cells,T_cells_c11_MKI67,19


# 3. 构建参考图谱和分层标签

选取作者已经构建好的CAFs/T细胞中细分小类的细胞标签，使用主要细胞类型作为参考，并将作者 T-cells 大类展开为
CD4 T、CD8 T、NK、NKT 和 Cycling T。


In [4]:
# 使用文章中给的第一大类
reference_label = adata_sc.obs["celltype_major"].astype(str).copy()

# T细胞的细分小类种类太多，有CD8和CD4，所以要选择T细胞的细分小类

t_groups = adata_sc.obs["celltype_major"].eq("T-cells")

reference_label.loc[t_groups] = adata_sc.obs.loc[t_groups, "celltype_minor"].astype("string")

adata_sc.obs["reference_label"] = reference_label.astype("category")

display(adata_sc.obs["reference_label"].value_counts().rename("n_cells").to_frame())

,n_cells
reference_label,
Cancer Epithelial,2223
PVL,592
Myeloid,255
T cells CD4+,239
Endothelial,219
CAFs,102
T cells CD8+,98
Plasmablasts,96
B-cells,56


# 4. 统计单细胞和空间组学中的共同基因

单细胞使用 layers["counts"]，空间数据使用尚未归一化的 X。
取两边共同的基因并统一排列顺序，构建新的参考对象与空间对象。

In [5]:
from scipy import sparse
import numpy as np
import anndata as ad

# 获取相同基因
common_genes = adata_sc.var_names[adata_sc.var_names.isin(adata_sp.var_names)]

print("共同基因数：", len(common_genes))

sc_aligned = adata_sc[:, common_genes].copy()
sp_aligned = adata_sp[:, common_genes].copy()

# 单细胞的参考基因：X使用原始counts
adata_ref = ad.AnnData(
    X=sc_aligned.layers["counts"].copy(),
    obs=sc_aligned.obs[
        [
            "orig.ident",
            "subtype",
            "celltype_major",
            "celltype_minor",
            "celltype_subset",
            "reference_label",
        ]
    ].copy(),
    var=sc_aligned.var[["gene_symbol"]].copy(),
)

# 空间对象保留各种obsm注释
adata_sp_input = sp_aligned

output_dir = project_dir / "data/processed/CID4535_combine_v1"
output_dir.mkdir(parents=True, exist_ok=True)

adata_ref.write_h5ad(
    output_dir / "single_cell_reference_counts.h5ad",
    compression="gzip",
)

adata_sp_input.write_h5ad(
    output_dir / "spatial_common_gene_counts.h5ad",
    compression="gzip",
)

# 保存标签数量和共同基因，方便核对
adata_ref.obs["reference_label"].value_counts().rename("n_cells").to_csv(output_dir / "reference_label_counts.csv")

adata_ref.var[["gene_symbol"]].to_csv(
    output_dir / "common_genes.csv",
    index_label="gene_id",
)

print("保存目录：", output_dir.resolve())

共同基因数： 15770
保存目录： /Users/georicl/Documents/sc_sncell_github/data/processed/CID4535_combine_v1


# 5. 保存 RCTD需要的矩阵和metadata

anndata的表达矩阵为 spot x 基因，RCTD则需要转置为基因 x 细胞

In [6]:
import pandas as pd
from scipy.io import mmwrite

input_dir = project_dir / "data/processed/CID4535_combine_v1"
export_dir = input_dir / "rctd_input"
export_dir.mkdir(parents=True, exist_ok=True)

# 导出矩阵和基因名
mmwrite(export_dir / "reference_counts.mtx", adata_ref.X.T)

mmwrite(
    export_dir / "spatial_counts.mtx",
    adata_sp_input.X.T,
)

pd.Series(adata_ref.var_names).to_csv(
    export_dir / "genes.tsv",
    sep="\t",
    index=False,
    header=False,
)

# 导出用于参考的细胞标签
reference_metadata = adata_ref.obs[["orig.ident", "reference_label"]].copy()

reference_metadata.index.name = "barcode"

reference_metadata.to_csv(export_dir / "reference_metadata.csv")

# 导出空间坐标
spatial_metadata = pd.DataFrame(
    adata_sp_input.obsm["spatial"],
    index=adata_sp_input.obs_names,
    columns=["x", "y"],
)

spatial_metadata.index.name = "barcode"

spatial_metadata.to_csv(
    export_dir / "spatial_metadata.csv"
)